# Unusual Observations and Influence

Outliers, high-leverage points, and influential points are related but not identical. This notebook shows how to compute the main diagnostics used in the lecture and software notes.

By the end of this notebook, you should be able to:

- distinguish outliers, high-leverage observations, and influential observations;
- compute hat values, studentized residuals, Cook's distance, DFFITS, and PRESS-related quantities;
- use thresholds as flags for investigation, not as automatic deletion rules;
- explain what to do after finding an unusual observation.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

obs = pd.read_csv(Path('data/unusual_observations.csv'))
model = smf.ols('Y ~ X', data=obs).fit()
print(model.summary().tables[1])

## Three Different Ideas

- Outlier: unusual in the response direction, often flagged by a large residual or studentized residual.
- High leverage: unusual in predictor space, flagged by a large hat value $h_i$.
- Influential: materially changes the fitted model, often flagged by Cook's distance or DFFITS.

An observation can be one, two, or all three. High leverage is not automatically influence; it becomes influential only when it materially changes fitted values, coefficients, or predictions.

In [ ]:
colors = obs['Case'].map({'Regular': 'tab:blue', 'Outlier': 'tab:orange', 'High leverage': 'tab:green', 'Influential': 'tab:red'})
plt.figure(figsize=(7, 4.2))
plt.scatter(obs['X'], obs['Y'], c=colors, s=55)
x_grid = np.linspace(obs['X'].min(), obs['X'].max(), 100)
plt.plot(x_grid, model.predict(pd.DataFrame({'X': x_grid})), color='black')
for _, row in obs.iterrows():
    if row['Case'] != 'Regular':
        plt.annotate(row['Case'], (row['X'], row['Y']), xytext=(5, 5), textcoords='offset points')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Outlier, leverage, and influence examples')
plt.tight_layout()

In [ ]:
infl = model.get_influence()
frame = infl.summary_frame()
result = obs.join(frame[['hat_diag', 'student_resid', 'cooks_d', 'dffits']])
result['abs_student_resid'] = result['student_resid'].abs()
result.sort_values('cooks_d', ascending=False).head(8)

## Rules of Thumb

Let $k$ be the number of predictors and $n$ the sample size.

- Studentized residual: $|t_i| > 2$ is a useful outlier flag; $|t_i| > 3$ is stronger evidence.
- High leverage: $h_i > 2(k+1)/n$ is a common flag.
- Cook's distance: compare values with each other; $D_i > 1$ or $D_i > 4/n$ are common flags.
- DFFITS: $|DFFITS_i| > 2\sqrt{(k+1)/(n-k-1)}$ flags possible influence.

These are screening thresholds. Investigate data quality and model form before removing observations.

In [ ]:
n = int(model.nobs)
k = int(model.df_model)
leverage_cutoff = 2 * (k + 1) / n
cook_cutoff = 4 / n
dffits_cutoff = 2 * np.sqrt((k + 1) / (n - k - 1))
flags = result.assign(
    flag_outlier=lambda d: d['abs_student_resid'] > 2,
    flag_leverage=lambda d: d['hat_diag'] > leverage_cutoff,
    flag_cook=lambda d: d['cooks_d'] > cook_cutoff,
    flag_dffits=lambda d: d['dffits'].abs() > dffits_cutoff,
)
print('leverage cutoff:', round(leverage_cutoff, 3))
print('Cook cutoff:', round(cook_cutoff, 3))
print('DFFITS cutoff:', round(dffits_cutoff, 3))
flags.loc[flags[['flag_outlier', 'flag_leverage', 'flag_cook', 'flag_dffits']].any(axis=1),
          ['X', 'Y', 'Case', 'student_resid', 'hat_diag', 'cooks_d', 'dffits', 'flag_outlier', 'flag_leverage', 'flag_cook', 'flag_dffits']]

## PRESS and Deleted Residual Logic

PRESS is the sum of squared deleted residuals. A deleted residual compares $y_i$ with the fitted value obtained from the model that left observation $i$ out.

For ordinary least squares, deleted residuals can be computed from ordinary residuals and leverage:

$$e_{(i)} = \frac{e_i}{1-h_i}.$$

In [ ]:
deleted_resid = model.resid / (1 - result['hat_diag'])
press = float(np.sum(deleted_resid ** 2))
print(f'PRESS: {press:.3f}')
pd.DataFrame({'Case': obs['Case'], 'deleted_residual': deleted_resid}).assign(abs_deleted=lambda d: d['deleted_residual'].abs()).sort_values('abs_deleted', ascending=False).head()

## Sensitivity Check

Influence is about change. After identifying a possible influential case, compare the fitted model with and without that case. A large coefficient change is evidence that the case deserves investigation and clear reporting.

In [ ]:
most_influential_index = result['cooks_d'].idxmax()
model_without = smf.ols('Y ~ X', data=obs.drop(index=most_influential_index)).fit()

comparison = pd.DataFrame({
    'full_model': model.params,
    'without_largest_cooks_d': model_without.params,
})
comparison['absolute_change'] = (comparison['without_largest_cooks_d'] - comparison['full_model']).abs()
print('Removed case:')
print(obs.loc[most_influential_index, ['X', 'Y', 'Case']])
comparison

## What to Do

Possible causes include data entry error, measurement error, a missing predictor, wrong model order, nonconstant variance, or a real but rare observation. Correct known mistakes. If the point is real, report sensitivity: compare the model with and without it, and explain which version answers the practical question.